In [0]:
from pyspark.sql.functions import col, to_timestamp, sum as spark_sum, when,row_number,count
from pyspark.sql import Window

In [0]:
CATALOG = "flightdata"
SCHEMA = "silver"

bronze_table = f"{CATALOG}.bronze.bronze_flights"
silver_table = f"{CATALOG}.{SCHEMA}.silver_flights"

In [0]:
sourceFlightDF = spark.table(bronze_table)

### FLATTEN NESTED STRUCTS

In [0]:
silverFlightDF = sourceFlightDF.select(
    col("flight_date"),
    col("flight_status"),
    col("ingestion_timestamp"),

    col("departure.airport").alias("departure_airport"),
    col("departure.timezone").alias("departure_timezone"),
    col("departure.iata").alias("departure_iata"),
    col("departure.icao").alias("departure_icao"),
    col("departure.terminal").alias("departure_terminal"),
    col("departure.gate").alias("departure_gate"),
    col("departure.delay").alias("departure_delay"),
    col("departure.scheduled").alias("departure_scheduled"),
    col("departure.estimated").alias("departure_estimated"),
    col("departure.actual").alias("departure_actual"),
    col("departure.estimated_runway").alias("departure_estimated_runway"),
    col("departure.actual_runway").alias("departure_actual_runway"),

    col("arrival.airport").alias("arrival_airport"),
    col("arrival.timezone").alias("arrival_timezone"),
    col("arrival.iata").alias("arrival_iata"),
    col("arrival.icao").alias("arrival_icao"),
    col("arrival.terminal").alias("arrival_terminal"),
    col("arrival.gate").alias("arrival_gate"),
    col("arrival.baggage").alias("arrival_baggage"),
    col("arrival.scheduled").alias("arrival_scheduled"),
    col("arrival.delay").alias("arrival_delay"),
    col("arrival.estimated").alias("arrival_estimated"),
    col("arrival.actual").alias("arrival_actual"),
    col("arrival.estimated_runway").alias("arrival_estimated_runway"),
    col("arrival.actual_runway").alias("arrival_actual_runway"),

    col("airline.name").alias("airline_name"),
    col("airline.iata").alias("airline_iata"),
    col("airline.icao").alias("airline_icao"),

    col("flight.number").alias("flight_number"),
    col("flight.iata").alias("flight_iata"),
    col("flight.icao").alias("flight_icao"),
    col("flight.codeshared").alias("flight_codeshared"),

    col("aircraft.registration").alias("aircraft_registration"),
    col("aircraft.iata").alias("aircraft_iata"),
    col("aircraft.icao").alias("aircraft_icao"),
    col("aircraft.icao24").alias("aircraft_icao24"),

    col("live"),
)

### CAST TIMESTAMP STRINGS

In [0]:
silverFlightDF = (silverFlightDF
    .withColumn("departure_scheduled", to_timestamp("departure_scheduled"))
    .withColumn("departure_estimated", to_timestamp("departure_estimated"))
    .withColumn("departure_actual", to_timestamp("departure_actual"))
    .withColumn("arrival_scheduled", to_timestamp("arrival_scheduled"))
    .withColumn("arrival_estimated", to_timestamp("arrival_estimated"))
    .withColumn("arrival_actual", to_timestamp("arrival_actual"))
)

In [0]:
(silverFlightDF
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("flight_date")
    .saveAsTable(silver_table))

In [0]:
silverFlightDF.printSchema()
display(spark.table(silver_table).limit(5))

###  FILL NULLS WITH DEFAULTS

In [0]:
string_defaults = {
    "flight_status": "unknown",
    "departure_terminal": "unknown",
    "departure_gate": "unknown",
    "arrival_terminal": "unknown",
    "arrival_gate": "unknown",
    "arrival_baggage": "unknown",
    "airline_name": "unknown",
    "airline_iata": "unknown",
    "airline_icao": "unknown",
    "flight_number": "unknown",
    "flight_iata": "unknown",
    "flight_icao": "unknown",
    "flight_codeshared": "none",
    "aircraft_registration": "unknown",
    "aircraft_iata": "unknown",
    "aircraft_icao": "unknown",
    "aircraft_icao24": "unknown",
    "live": "unknown",
}

int_defaults = {
    "departure_delay": 0,
    "arrival_delay": 0,
}


silverFlightDF = silverFlightDF.na.fill(string_defaults)
silverFlightDF = silverFlightDF.na.fill(int_defaults)

In [0]:
(silverFlightDF
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("flight_date")
    .saveAsTable(silver_table))

silverFlightDF.printSchema()
display(spark.table(silver_table).limit(5))

###  DROP COLUMN THAT IS ENTIRELY NULL

In [0]:
total_rows = silverFlightDF.count()

# Compute null count per column
null_counts_row = silverFlightDF.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in silverFlightDF.columns
]).collect()[0]

columns_to_drop = [
    c for c in silverFlightDF.columns
    if (null_counts_row[c] / total_rows) > 0.70
]

print(f"Total rows: {total_rows}")
print("Columns exceeding 70% missing (will be dropped):")
for c in columns_to_drop:
    pct = (null_counts_row[c] / total_rows) * 100
    print(f"  - {c}: {pct:.1f}% missing")

silverFlightDF = silverFlightDF.drop(*columns_to_drop)

print(f"\nRemaining columns: {len(silverFlightDF.columns)}")


In [0]:
silverFlightDF = silverFlightDF.drop(*columns_to_drop)

print(f"Remaining columns: {len(silverFlightDF.columns)}")
silverFlightDF.printSchema()

In [0]:
silverFlightDF = silverFlightDF.filter(
    col("arrival_airport").isNotNull() & col("arrival_timezone").isNotNull()
)

In [0]:
(silverFlightDF
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("flight_date")
    .saveAsTable(silver_table))

In [0]:
null_counts = silverFlightDF.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in silverFlightDF.columns
])
display(null_counts)

In [0]:

print(f"Row count after cleaning: {silverFlightDF.count()}")

In [0]:
bad_row_count = silverFlightDF.filter(col("departure_iata") == col("arrival_iata")).count()
if bad_row_count > 0:
    print(f"Dropping {bad_row_count} row(s) with identical departure/arrival airport (bad data).")

silverFlightDF = silverFlightDF.filter(
    col("departure_iata") != col("arrival_iata")
)

### DEDUPLICATE

In [0]:
dedup_key = ["flight_iata", "departure_iata", "departure_scheduled"]

window_spec = Window.partitionBy(*dedup_key).orderBy(col("ingestion_timestamp").desc())

silverFlightDF = (silverFlightDF
    .withColumn("_row_num", row_number().over(window_spec))
    .filter(col("_row_num") == 1)
    .drop("_row_num")
)

print(f"Row count after deduplication: {silverFlightDF.count()}")

In [0]:
dup_check = (silverFlightDF
    .groupBy("flight_iata", "departure_iata", "departure_scheduled")
    .agg(count("*").alias("cnt"))
    .filter(col("cnt") > 1))

display(dup_check)
print(f"Remaining duplicate groups: {dup_check.count()}")

In [0]:
(silverFlightDF
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("flight_date")
    .saveAsTable(silver_table))

In [0]:
print(f"Silver table written: {silver_table}")
display(spark.table(silver_table).limit(5))

### Data Quality

In [0]:
dq_failures = []

In [0]:
# 1. Row count sanity check — should never be empty
row_count = silverFlightDF.count()
if row_count == 0:
    dq_failures.append("Row count is 0 — no data to write.")

In [0]:
# 2. Critical fields must never be null
critical_cols = ["flight_date", "departure_iata", "arrival_iata", "departure_scheduled"]
for c in critical_cols:
    null_count = silverFlightDF.filter(col(c).isNull()).count()
    if null_count > 0:
        dq_failures.append(f"Column '{c}' has {null_count} null value(s) — expected 0.")


In [0]:
# 3. flight_date should be a valid, parseable date (not garbage strings)
bad_dates = silverFlightDF.filter(
    ~col("flight_date").rlike(r"^\d{4}-\d{2}-\d{2}$")
).count()
if bad_dates > 0:
    dq_failures.append(f"{bad_dates} row(s) have a malformed flight_date.")


In [0]:
# 4. Deduplication check — no duplicate flight instances should remain
dup_count = (silverFlightDF
    .groupBy("flight_iata", "departure_iata", "departure_scheduled")
    .agg(count("*").alias("cnt"))
    .filter(col("cnt") > 1)
    .count())
if dup_count > 0:
    dq_failures.append(f"{dup_count} duplicate flight group(s) still present after dedup.")


In [0]:
# 5. Departure/arrival airports shouldn't be identical (data corruption signal)
same_airport = silverFlightDF.filter(
    col("departure_iata") == col("arrival_iata")
).count()
if same_airport > 0:
    dq_failures.append(f"{same_airport} row(s) have identical departure and arrival airports.")


In [0]:
if dq_failures:
    print("DATA QUALITY CHECKS FAILED:")
    for failure in dq_failures:
        print(f"  - {failure}")
    raise ValueError(f"{len(dq_failures)} data quality check(s) failed. See output above.")
else:
    print(f"All data quality checks passed. {row_count} rows validated.")

In [0]:
(silverFlightDF
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("flight_date")
    .saveAsTable(silver_table))

print(f"Silver table written: {silver_table}")
display(spark.table(silver_table).limit(5))